## Cell 1: Feedforward neural network for building energy use

**Owner: Alex Mwamsindo**

> **Version v06 — hyperparameter tuning on the v04 configuration**
> Inputs and target: identical to v04 (183 columns with log-scaled products, log1p target).
> The **Experiment configuration** and **Build, train, and evaluate the network** sections
> rerun v04's settings as `v06_reference` (a reproducibility check).
> The **Hyperparameter grid** section then tests 12 training/architecture settings,
> each with 3 seeds.
> Purpose: address the early peaking (best epochs 4–9) seen in all log-target versions
> and select the best settings by validation MAE.

#### Objectives
1. Build a reproducible feedforward neural network in TensorFlow/Keras to predict
   `MFBTU`, annual major-fuels consumption in kBtu.
2. Find the input configuration that gives the best validation performance by
   comparing the base feature set (175 columns) with the engineered set (183 columns),
   while keeping all 23 source variables.
3. Compare a raw `MFBTU` target with a `log1p(MFBTU)` target, scoring both in original kBtu.
4. Tune a small, justified set of hyperparameters (layers, units, learning rate,
   batch size, dropout, L2 regularization, early stopping) and check for overfitting
   with learning curves.
5. Evaluate the selected model across building subgroups, then score it once on
   the held-out test set.

#### Motivation
Commercial real estate acquisition analysts often need an energy estimate before
they have a building's utility history. A neural network can learn nonlinear
relationships and interactions between building size, operations, equipment and
climate that a linear model cannot represent directly. We test whether that
flexibility improves on the team's linear regression baseline and how it compares
with gradient boosting, without assuming it will.

#### Data
The team's processed 2018 CBECS file (`cbecs2018_processed.csv`, 6,357 buildings):
23 source variables with missing-value flags, survey-cap flags, one-hot encoded
categories, and eight candidate engineered features. We keep the project's fixed
70/15/15 split (4,449 / 954 / 954). Scaling is fitted on training rows only.
Validation MAE in kBtu is the main selection metric, as in the gradient boosting
notebook. The test set is used once, after model selection is complete.

In [1]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 3: Imports, package versions, and reproducibility settings
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
tf.keras.utils.set_random_seed(SEED)            # seeds Python, NumPy, and TensorFlow together
tf.config.experimental.enable_op_determinism()  # makes TF operations repeatable (TF >= 2.9)

versions = {'Python': platform.python_version(), 'NumPy': np.__version__, 'pandas': pd.__version__,
            'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__,
            'GPUs visible': len(tf.config.list_physical_devices('GPU'))}
display(pd.Series(versions, name='Version').to_frame())

,Version
Python,3.13.18
NumPy,2.4.6
pandas,2.2.3
scikit-learn,1.6.1
TensorFlow,2.21.0
GPUs visible,0


In [3]:
# Cell 4: Load the processed data and run integrity checks
data_path = Path('/content/drive/MyDrive/Data/cbecs2018_processed.csv')
data = pd.read_csv(data_path)

ID_COLUMNS = ['PUBID', 'split', 'MFBTU']
NUMERIC = ['SQFT', 'FLCEILHT', 'WKHRS', 'NWKER', 'MONUSE', 'HDD65', 'CDD65', 'HEATP', 'COOLP', 'OCCUPYP']
ENGINEERED = ['log1p_SQFT', 'log1p_NWKER', 'workers_per_1000_sqft', 'weekly_floor_area_hours',
              'heated_floor_area', 'cooled_floor_area', 'heating_area_degree_days', 'cooling_area_degree_days']
all_features = [c for c in data.columns if c not in ID_COLUMNS]
binary_features = [c for c in all_features if c not in NUMERIC + ENGINEERED]
feature_sets = {'base': [c for c in all_features if c not in ENGINEERED], 'engineered': all_features}

# Structure and split checks
assert data.shape == (6357, 186), data.shape
assert data.PUBID.is_unique and data.PUBID.notna().all()
assert data['split'].value_counts().to_dict() == {'train': 4449, 'validation': 954, 'test': 954}
assert len(feature_sets['base']) == 175 and len(feature_sets['engineered']) == 183
# Leakage and value checks
assert not set(ID_COLUMNS) & set(all_features), 'ID, split, or target found among features'
assert np.isfinite(data[all_features + ['MFBTU']].to_numpy(dtype=float)).all()
assert data.MFBTU.gt(0).all(), 'log1p target requires positive MFBTU'
assert data[binary_features].isin([0, 1]).all().all()
assert data.SQFT.le(1_000_000).all() and data.NWKER.le(5_000).all()

# Same row order convention as the gradient boosting notebook
train = data.loc[data['split'].eq('train')].sort_values('PUBID').reset_index(drop=True)
validation = data.loc[data['split'].eq('validation')].sort_values('PUBID').reset_index(drop=True)
n_test = int(data['split'].eq('test').sum())   # test rows stay untouched until the final stage

display(pd.DataFrame({'Rows': [len(train), len(validation), n_test]},
                     index=['train', 'validation', 'test (sealed)']))
display(pd.DataFrame({'Columns': [len(NUMERIC), len(ENGINEERED), len(binary_features),
                                  len(feature_sets['base']), len(feature_sets['engineered'])]},
                     index=['Numeric source', 'Engineered', 'Binary (flags + one-hot)', 'Base total', 'Engineered total']))
print(f"Training MFBTU: median {train.MFBTU.median():,.0f} kBtu, mean {train.MFBTU.mean():,.0f} kBtu")
print(f"Training skewness: raw {train.MFBTU.skew():.2f}, log1p {np.log1p(train.MFBTU).skew():.2f}")
print('All Stage 1 integrity checks passed.')

,Rows
train,4449
validation,954
test (sealed),954


,Columns
Numeric source,10
Engineered,8
Binary (flags + one-hot),165
Base total,175
Engineered total,183


Training MFBTU: median 4,397,988 kBtu, mean 15,868,331 kBtu
Training skewness: raw 7.69, log1p -0.42
All Stage 1 integrity checks passed.


## Cell 5: Experiment configuration

Every choice that differs between versions lives in the `CONFIG` cell below:
feature set, target transform, input transformations, architecture, and training
settings. All other cells are identical across v01–v06, so differences in results
can be attributed to the configuration alone.

Early stopping monitors a fixed 15% of the training rows (seed 42), never the
official validation set. The validation set is used only to compare versions,
and the test set stays sealed. Each version is trained with three random seeds
so we can separate real improvements from run-to-run noise.

In [4]:
# Cell 6: Experiment configuration (the only code cell that changes between versions)
CONFIG = {
    'version': 'v06_reference',       # v04 settings; the grid in Cell 14 varies training/architecture only
    # Inputs
    'feature_set': 'engineered',      # 'base' (175 columns) or 'engineered' (183 columns)
    'log_heavy_features': True,       # v04: log1p the heavy-tailed engineered products before scaling
    'drop_sources': [],               # v05: source variables to remove (with their flags/one-hot columns)
    'drop_columns': [],               # v05: individual columns to remove (e.g., engineered features built from dropped inputs)
    # Target
    'target': 'log1p',                # 'log1p' or 'raw'; both are standardized and scored in kBtu
    # Architecture
    'hidden_units': [64, 32],
    'activation': 'relu',
    'dropout': 0.0,
    'l2': 0.0,
    # Training
    'learning_rate': 1e-3,
    'batch_size': 32,
    'max_epochs': 300,
    'patience': 20,                   # epochs without monitor improvement before stopping
    'monitor_fraction': 0.15,
    'seeds': [42, 43, 44],
}
display(pd.Series({k: str(v) for k, v in CONFIG.items()}, name='Setting').to_frame())

,Setting
version,v06_reference
feature_set,engineered
log_heavy_features,True
drop_sources,[]
drop_columns,[]
target,log1p
hidden_units,"[64, 32]"
activation,relu
dropout,0.0
l2,0.0


In [5]:
# Cell 7: Preparing inputs and target for this version
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

HEAVY_TAILED = ['weekly_floor_area_hours', 'heated_floor_area', 'cooled_floor_area',
                'heating_area_degree_days', 'cooling_area_degree_days']

# 1. Columns used in this version
feature_columns = [c for c in feature_sets[CONFIG['feature_set']]
                   if not any(c == s or c.startswith(s + '__') for s in CONFIG['drop_sources'])
                   and c not in CONFIG['drop_columns']]
continuous_columns = [c for c in feature_columns if c in NUMERIC + ENGINEERED]
binary_columns = [c for c in feature_columns if c not in continuous_columns]

# 2. Fixed early-stopping rows carved from training (identical in every version)
fit_idx, monitor_idx = train_test_split(train.index, test_size=CONFIG['monitor_fraction'], random_state=SEED)
fit, monitor = train.loc[sorted(fit_idx)], train.loc[sorted(monitor_idx)]
frames = {'fit': fit, 'monitor': monitor, 'validation': validation}

# 3. Inputs: optional log of heavy tails, then standardize continuous columns on fit rows only
def transform_inputs(frame):
    values = frame[feature_columns].astype(float).copy()
    if CONFIG['log_heavy_features']:
        cols = [c for c in HEAVY_TAILED if c in values]
        values[cols] = np.log1p(values[cols])
    return values

X_frames = {name: transform_inputs(f) for name, f in frames.items()}
scaler = StandardScaler().fit(X_frames['fit'][continuous_columns])
for name in X_frames:
    X_frames[name][continuous_columns] = scaler.transform(X_frames[name][continuous_columns])
X = {name: f.to_numpy(dtype='float32') for name, f in X_frames.items()}

# 4. Target: log1p or raw, then standardized with fit-row statistics
def to_fit_scale(mfbtu):
    mfbtu = np.asarray(mfbtu, dtype=float)
    return np.log1p(mfbtu) if CONFIG['target'] == 'log1p' else mfbtu

t_fit = to_fit_scale(fit.MFBTU)
y_mean, y_std = t_fit.mean(), t_fit.std()

def encode_target(mfbtu):
    return ((to_fit_scale(mfbtu) - y_mean) / y_std).astype('float32')

def decode_to_kbtu(z):
    """Network output -> kBtu, clipped at 0 (team convention)."""
    t = np.asarray(z, dtype=float).ravel() * y_std + y_mean
    return np.maximum(np.expm1(t) if CONFIG['target'] == 'log1p' else t, 0)

y = {name: encode_target(f.MFBTU) for name, f in frames.items()}
y_kbtu = {name: f.MFBTU.to_numpy(dtype=float) for name, f in frames.items()}

# 5. Checks
assert len(fit) + len(monitor) == len(train) and set(fit.PUBID).isdisjoint(monitor.PUBID)
assert not set(ID_COLUMNS) & set(feature_columns)
assert all(np.isfinite(a).all() for a in X.values()) and all(np.isfinite(a).all() for a in y.values())
assert np.allclose(X_frames['fit'][continuous_columns].mean(), 0, atol=1e-8)
assert np.allclose(X_frames['fit'][continuous_columns].std(ddof=0), 1, atol=1e-8)
assert np.allclose(decode_to_kbtu(y['fit']), y_kbtu['fit'], rtol=1e-4, atol=10), 'target round trip failed'

display(pd.DataFrame({'Rows': [len(f) for f in frames.values()], 'Columns': [X[n].shape[1] for n in X]},
                     index=['fit (trains weights)', 'monitor (early stopping)', 'validation (compares versions)']))
print(f"{CONFIG['version']}: {len(feature_columns)} inputs "
      f"({len(continuous_columns)} scaled continuous, {len(binary_columns)} binary 0/1)")
print(f"Target '{CONFIG['target']}' standardized with fit-row mean {y_mean:.4f}, std {y_std:.4f}")
print('Stage 2 checks passed.')

,Rows,Columns
fit (trains weights),3781,183
monitor (early stopping),668,183
validation (compares versions),954,183


v06_reference: 183 inputs (18 scaled continuous, 165 binary 0/1)
Target 'log1p' standardized with fit-row mean 14.9487, std 2.1135
Stage 2 checks passed.


## Cell 8: Build, train, and evaluate the network

The network is fully connected: inputs → Dense(64, ReLU) → Dense(32, ReLU) →
Dense(1, linear). It minimizes mean squared error on the standardized target (log1p or raw, as set in
`CONFIG`) with the Adam optimizer. Training stops when the monitor-set loss has not improved
for `patience` epochs, and the weights from the best epoch are restored.

Each version is trained with three seeds. We report validation MAE, RMSE, and R²
in kBtu as the mean ± standard deviation across seeds, so a later version only
counts as better if its gain is larger than this run-to-run variation.

In [6]:
# Cell 9: Model builder, metrics, and training function
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def build_model(n_inputs, cfg):
    reg = tf.keras.regularizers.l2(cfg['l2']) if cfg['l2'] > 0 else None
    model = tf.keras.Sequential([tf.keras.Input(shape=(n_inputs,))])
    for units in cfg['hidden_units']:
        model.add(tf.keras.layers.Dense(units, activation=cfg['activation'], kernel_regularizer=reg))
        if cfg['dropout'] > 0:
            model.add(tf.keras.layers.Dropout(cfg['dropout']))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['learning_rate']), loss='mse')
    return model

def kbtu_metrics(actual, predicted):
    return {'MAE_kBtu': mean_absolute_error(actual, predicted),
            'RMSE_kBtu': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

def train_one_seed(seed, cfg):
    tf.keras.utils.set_random_seed(seed)
    model = build_model(X['fit'].shape[1], cfg)
    stopper = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg['patience'],
                                               restore_best_weights=True)

    history = model.fit(X['fit'], y['fit'], validation_data=(X['monitor'], y['monitor']),
                        epochs=cfg['max_epochs'], batch_size=cfg['batch_size'],
                        shuffle=True, verbose=0, callbacks=[stopper])
    preds = {name: decode_to_kbtu(model.predict(X[name], verbose=0)) for name in X}
    return model, history.history, preds

build_model(X['fit'].shape[1], CONFIG).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        11,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,889 (54.25 KB)

 Trainable params: 13,889 (54.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Cell 10: Train with each seed and evaluate in kBtu
import time

runs, rows = {}, []
max_train_kbtu = train.MFBTU.max()
for seed in CONFIG['seeds']:
    start = time.time()
    model, hist, preds = train_one_seed(seed, CONFIG)
    best_epoch = int(np.argmin(hist['val_loss'])) + 1
    runs[seed] = {'model': model, 'history': hist, 'preds': preds, 'best_epoch': best_epoch}
    for name in ['fit', 'monitor', 'validation']:
        assert np.isfinite(preds[name]).all(), f'Non-finite predictions ({name}, seed {seed})'
        rows.append({'seed': seed, 'partition': name, **kbtu_metrics(y_kbtu[name], preds[name]),
                     'max_pred_kBtu': preds[name].max(),
                     'n_above_train_max': int((preds[name] > max_train_kbtu).sum())})
    print(f'Seed {seed}: ran {len(hist["loss"])} epochs, best epoch {best_epoch}, {time.time() - start:.0f}s')

seed_results = pd.DataFrame(rows)
summary = seed_results.groupby('partition', sort=False)[['MAE_kBtu', 'RMSE_kBtu', 'R2']].agg(['mean', 'std'])
display(seed_results.round(3))
display(summary.round(3))

Seed 42: ran 25 epochs, best epoch 5, 13s
Seed 43: ran 26 epochs, best epoch 6, 13s
Seed 44: ran 25 epochs, best epoch 5, 14s


,seed,partition,MAE_kBtu,RMSE_kBtu,R2,max_pred_kBtu,n_above_train_max
0,42,fit,4816451.136,1.812322e+07,0.770,3.870113e+08,0
1,42,monitor,5792067.976,2.175533e+07,0.716,2.678329e+08,0
2,42,validation,4838394.735,1.629021e+07,0.801,2.993924e+08,0
3,43,fit,4575880.280,1.781320e+07,0.778,3.111785e+08,0
4,43,monitor,6237713.626,2.222048e+07,0.704,3.227066e+08,0
5,43,validation,5149401.400,1.820977e+07,0.751,2.680088e+08,0
6,44,fit,5245902.203,1.825262e+07,0.767,7.011480e+08,0
7,44,monitor,5920311.644,1.743788e+07,0.818,4.106966e+08,0
8,44,validation,5332969.744,1.637773e+07,0.798,3.638399e+08,0


MAE_kBtu                 RMSE_kBtu                  R2       
                   mean         std          mean          std   mean    std
partition                                                                   
fit         4879411.206  339419.095  1.806301e+07   225813.979  0.771  0.006
monitor     5983364.415  229416.107  2.047123e+07  2637234.228  0.746  0.062
validation  5106921.960  250008.974  1.695923e+07  1083877.022  0.783  0.028

In [8]:
# Cell 11: Record this version in the shared experiment log
import json
from datetime import datetime

LOG_PATH = data_path.parent / 'nn_experiment_log.csv'
entry = {'version': CONFIG['version'], 'timestamp': datetime.now().isoformat(timespec='seconds'),
         'n_inputs': len(feature_columns),
         'mean_best_epoch': float(np.mean([r['best_epoch'] for r in runs.values()]))}
for part, prefix in [('validation', 'val'), ('fit', 'fit')]:
    part_rows = seed_results[seed_results.partition.eq(part)]
    for metric in ['MAE_kBtu', 'RMSE_kBtu', 'R2']:
        entry[f'{prefix}_{metric}_mean'] = part_rows[metric].mean()
        entry[f'{prefix}_{metric}_std'] = part_rows[metric].std()
entry['config'] = json.dumps(CONFIG)

log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
if not log.empty:
    log = log[log['version'].ne(CONFIG['version'])]
log = pd.concat([log, pd.DataFrame([entry])], ignore_index=True)
log.to_csv(LOG_PATH, index=False)
display(log.drop(columns='config').round(3))
print(f'Logged {CONFIG["version"]} to {LOG_PATH}')

,version,timestamp,n_inputs,mean_best_epoch,val_MAE_kBtu_mean,val_MAE_kBtu_std,val_RMSE_kBtu_mean,val_RMSE_kBtu_std,val_R2_mean,val_R2_std,fit_MAE_kBtu_mean,fit_MAE_kBtu_std,fit_RMSE_kBtu_mean,fit_RMSE_kBtu_std,fit_R2_mean,fit_R2_std
0,v01_base_log,2026-11-14T23:53:01,175,8.000,5237602.429,181773.519,1.654535e+07,1852171.543,0.793,0.046,4937323.867,97952.862,1.814874e+07,608745.647,0.769,0.015
1,v02_engineered_log,2026-11-15T00:04:09,183,5.000,5207122.898,396135.435,1.703522e+07,402956.017,0.782,0.010,5177520.145,540436.732,1.972582e+07,3405880.332,0.722,0.098
2,v03_raw_target,2026-11-15T00:15:21,175,18.667,6251088.435,770918.247,1.507577e+07,722011.946,0.829,0.016,4227326.028,935832.973,9.244136e+06,2125515.051,0.938,0.026
3,v04_logscaled_products,2026-11-15T00:31:26,183,5.333,5106921.960,250008.974,1.695923e+07,1083877.022,0.783,0.028,4879411.206,339419.095,1.806301e+07,225813.979,0.771,0.006
4,v05_property_only,2026-11-15T00:53:18,117,8.000,5794869.399,50574.426,1.929038e+07,949415.679,0.720,0.028,5511318.660,41827.290,1.966145e+07,324123.936,0.729,0.009
5,v06_g01,2026-11-15T00:54:40,117,23.667,5909564.794,91016.838,1.963028e+07,436081.951,0.710,0.013,5394086.873,39892.157,1.927794e+07,429522.142,0.740,0.012
6,v06_g02,2026-11-15T00:56:35,117,68.333,5954768.259,12352.526,1.985283e+07,542161.538,0.704,0.016,5289815.884,133262.855,1.881394e+07,729735.170,0.752,0.019
7,v06_g03,2026-11-15T00:58:36,117,66.333,5856356.051,117002.494,2.043444e+07,369888.311,0.686,0.011,5381113.105,412788.235,1.992319e+07,1070716.156,0.721,0.029
8,v06_g04,2026-11-15T00:59:44,117,26.000,5851139.680,113457.529,1.934360e+07,348822.823,0.719,0.010,5292647.770,65966.476,1.901730e+07,528572.596,0.746,0.014
9,v06_g05,2026-11-15T01:01:21,117,50.000,5717045.978,124690.727,1.946162e+07,918498.547,0.715,0.027,4896951.709,111499.554,1.810453e+07,779241.704,0.770,0.020


Logged v06_reference to /content/drive/MyDrive/Data/nn_experiment_log.csv


## Cell 12: v06 results

Rerunning v04's settings as `v06_reference` reproduced v04 exactly (validation MAE
5,106,922 kBtu), confirming the notebooks are reproducible.

In the 12-setting grid, 11 settings outperformed the reference. Lowering the learning
rate from 0.001 to 0.0003 or 0.0001 was the main driver: best epochs moved from 5–6
to 16–117, and validation R² rose from 0.783 to about 0.81. The five best settings
were effectively tied (MAE 4.88M–4.94M). The best, `v06_g09` (learning rate 0.0003,
hidden layers [32, 16]), reached validation MAE 4.88M ± 0.08M kBtu, RMSE 15.9M kBtu,
and R² 0.810. Its 0.23M gain over the reference is below the 0.3M noise threshold on
its own, but the consistency across settings supports the effect of the lower
learning rate.

Dropout did not improve MAE and worsened RMSE and R². The three-layer network with
dropout (`v06_g11`) underfit (training MAE 7.23M, worse than validation), showing
that too much constraint is as harmful as too little. `v06_g09` was selected for its
lowest MAE, lowest seed variation, and smallest network (about 6,400 parameters).
Because it was chosen from 12 settings on the validation set, its validation score
is likely slightly optimistic.

## Cell 13: Hyperparameter grid

Every log-target version reached its best monitor loss within 4–9 epochs and then
began overfitting, which suggests the default learning rate (0.001) and the absence
of regularization limit performance. The grid below tests 12 settings chosen to
address this, rather than an exhaustive search:

- **Learning rate** (3e-4, 1e-4): smaller steps for more gradual training.
- **Dropout** (0.2) and **L2** (1e-4, 1e-3): regularization against overfitting.
- **Batch size** (64, 128): smoother gradient estimates.
- **Width and depth** ([32, 16], [128, 64], [64, 32, 16]): smaller and larger networks.

Inputs and target are fixed at v04's configuration. Each setting uses three seeds,
is logged as `v06_gNN`, and is ranked by validation MAE (RMSE breaks ties). A setting
counts as an improvement only if it beats the reference by more than about 0.3M kBtu.

In [10]:
# Cell 14: Hyperparameter grid
import json
import time

GRID = [
    {'learning_rate': 3e-4},
    {'learning_rate': 1e-4},
    {'learning_rate': 3e-4, 'dropout': 0.2},
    {'learning_rate': 3e-4, 'l2': 1e-4},
    {'learning_rate': 3e-4, 'l2': 1e-3},
    {'learning_rate': 3e-4, 'dropout': 0.2, 'l2': 1e-4},
    {'learning_rate': 3e-4, 'batch_size': 64},
    {'learning_rate': 3e-4, 'batch_size': 128},
    {'learning_rate': 3e-4, 'hidden_units': [32, 16]},
    {'learning_rate': 3e-4, 'hidden_units': [128, 64], 'dropout': 0.2},
    {'learning_rate': 3e-4, 'hidden_units': [64, 32, 16], 'dropout': 0.2},
    {'learning_rate': 1e-4, 'dropout': 0.2, 'l2': 1e-4},
]
TUNED_KEYS = ['hidden_units', 'dropout', 'l2', 'learning_rate', 'batch_size']
assert all(set(g) <= set(TUNED_KEYS) for g in GRID), 'The grid may only change training/architecture settings'
RESUME = True
assert CONFIG['version'] == 'v06_reference' and CONFIG['log_heavy_features'] and len(feature_columns) == 183, \
    'Grid must run on the v04 inputs: check the Experiment configuration cell'

def evaluate_config(cfg):
    rows, epochs = [], []
    for seed in cfg['seeds']:
        _, hist, preds = train_one_seed(seed, cfg)
        epochs.append(int(np.argmin(hist['val_loss'])) + 1)
        for name in ['fit', 'monitor', 'validation']:
            assert np.isfinite(preds[name]).all(), f'Non-finite predictions ({cfg["version"]}, {name}, seed {seed})'
            rows.append({'seed': seed, 'partition': name, **kbtu_metrics(y_kbtu[name], preds[name]),
                         'n_above_train_max': int((preds[name] > max_train_kbtu).sum())})
    return pd.DataFrame(rows), epochs

def log_config(cfg, results, epochs):
    entry = {'version': cfg['version'], 'timestamp': datetime.now().isoformat(timespec='seconds'),
             'n_inputs': len(feature_columns), 'mean_best_epoch': float(np.mean(epochs))}
    for part, prefix in [('validation', 'val'), ('fit', 'fit')]:
        part_rows = results[results.partition.eq(part)]
        for metric in ['MAE_kBtu', 'RMSE_kBtu', 'R2']:
            entry[f'{prefix}_{metric}_mean'] = part_rows[metric].mean()
            entry[f'{prefix}_{metric}_std'] = part_rows[metric].std()
    entry['config'] = json.dumps(cfg)
    log = pd.read_csv(LOG_PATH)
    log = log[log['version'].ne(cfg['version'])]
    pd.concat([log, pd.DataFrame([entry])], ignore_index=True).to_csv(LOG_PATH, index=False)

for i, overrides in enumerate(GRID, 1):
    cfg = {**CONFIG, **overrides, 'version': f'v06_g{i:02d}'}
    if RESUME and cfg['version'] in pd.read_csv(LOG_PATH)['version'].values:
        print(f'{cfg["version"]}: already logged, skipping')
        continue
    start = time.time()
    results, epochs = evaluate_config(cfg)
    log_config(cfg, results, epochs)
    val = results[results.partition.eq('validation')]
    print(f'{cfg["version"]} {overrides}: val MAE {val.MAE_kBtu.mean() / 1e6:.2f}M ± {val.MAE_kBtu.std() / 1e6:.2f}M, '
          f'best epochs {epochs}, above-max predictions {int(results.n_above_train_max.sum())}, '
          f'{time.time() - start:.0f}s')


log = pd.read_csv(LOG_PATH)
v06 = log[log['version'].str.startswith('v06_')].reset_index(drop=True)
params = v06['config'].map(json.loads).apply(pd.Series)[TUNED_KEYS]
table = pd.concat([v06[['version']], params,
                   v06[['mean_best_epoch', 'val_MAE_kBtu_mean', 'val_MAE_kBtu_std',
                        'val_RMSE_kBtu_mean', 'val_R2_mean', 'fit_MAE_kBtu_mean']]], axis=1)
table = table.sort_values(['val_MAE_kBtu_mean', 'val_RMSE_kBtu_mean']).reset_index(drop=True)
display(table.round({'mean_best_epoch': 1, 'val_MAE_kBtu_mean': 0, 'val_MAE_kBtu_std': 0,
                     'val_RMSE_kBtu_mean': 0, 'val_R2_mean': 3, 'fit_MAE_kBtu_mean': 0}))

best = table.iloc[0]
reference = table[table['version'].eq(CONFIG['version'])].iloc[0]
gain = reference.val_MAE_kBtu_mean - best.val_MAE_kBtu_mean
print(f'Best: {best.version}, val MAE {best.val_MAE_kBtu_mean / 1e6:.2f}M vs reference '
      f'{reference.val_MAE_kBtu_mean / 1e6:.2f}M (gain {gain / 1e6:.2f}M; '
      f'{"exceeds" if gain > 3e5 else "within"} the 0.3M noise threshold)')

v06_g01 {'learning_rate': 0.0003}: val MAE 5.01M ± 0.16M, best epochs [16, 18, 20], above-max predictions 0, 57s
v06_g02 {'learning_rate': 0.0001}: val MAE 4.89M ± 0.17M, best epochs [41, 46, 49], above-max predictions 0, 91s
v06_g03 {'learning_rate': 0.0003, 'dropout': 0.2}: val MAE 5.04M ± 0.18M, best epochs [35, 51, 52], above-max predictions 0, 100s
v06_g04 {'learning_rate': 0.0003, 'l2': 0.0001}: val MAE 4.94M ± 0.15M, best epochs [18, 17, 17], above-max predictions 0, 57s
v06_g05 {'learning_rate': 0.0003, 'l2': 0.001}: val MAE 5.07M ± 0.08M, best epochs [49, 53, 47], above-max predictions 0, 98s
v06_g06 {'learning_rate': 0.0003, 'dropout': 0.2, 'l2': 0.0001}: val MAE 4.96M ± 0.17M, best epochs [35, 61, 52], above-max predictions 0, 100s
v06_g07 {'learning_rate': 0.0003, 'batch_size': 64}: val MAE 4.92M ± 0.18M, best epochs [25, 25, 26], above-max predictions 0, 44s
v06_g08 {'learning_rate': 0.0003, 'batch_size': 128}: val MAE 4.91M ± 0.17M, best epochs [39, 35, 40], above-max pre

,version,hidden_units,dropout,l2,learning_rate,batch_size,mean_best_epoch,val_MAE_kBtu_mean,val_MAE_kBtu_std,val_RMSE_kBtu_mean,val_R2_mean,fit_MAE_kBtu_mean
0,v06_g09,"[32, 16]",0.0,0.0000,0.0003,32,23.3,4877769.0,79179.0,15904272.0,0.810,4679097.0
1,v06_g02,"[64, 32]",0.0,0.0000,0.0001,32,45.3,4893101.0,173046.0,15737106.0,0.813,4366378.0
2,v06_g08,"[64, 32]",0.0,0.0000,0.0003,128,38.0,4912468.0,167124.0,15541795.0,0.817,4470069.0
3,v06_g07,"[64, 32]",0.0,0.0000,0.0003,64,25.3,4917932.0,178686.0,15660089.0,0.815,4443332.0
4,v06_g04,"[64, 32]",0.0,0.0001,0.0003,32,17.3,4938799.0,148486.0,16046268.0,0.806,4482663.0
5,v06_g06,"[64, 32]",0.2,0.0001,0.0003,32,49.3,4959295.0,170414.0,17676699.0,0.765,4798518.0
6,v06_g10,"[128, 64]",0.2,0.0000,0.0003,32,35.3,4994413.0,165460.0,17181243.0,0.778,4350369.0
7,v06_g12,"[64, 32]",0.2,0.0001,0.0001,32,115.0,4995669.0,88930.0,17842822.0,0.761,4890528.0
8,v06_g01,"[64, 32]",0.0,0.0000,0.0003,32,18.0,5005176.0,161962.0,16249898.0,0.801,4433763.0
9,v06_g03,"[64, 32]",0.2,0.0000,0.0003,32,46.0,5038205.0,182460.0,17800424.0,0.761,4898059.0


Best: v06_g09, val MAE 4.88M vs reference 5.11M (gain 0.23M; within the 0.3M noise threshold)
